# Reading and Writing ONNX Models — Hands-On Application

## Objective

Master the complete ONNX model I/O lifecycle: serialization, file persistence, external data management,
memory-efficient loading, metadata inspection, and size analysis.

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Imports, helpers, and test model factory |
| 2 | [Exercise 1: Build and Save Models](#2-exercise-1) | Construct models and persist to disk |
| 3 | [Exercise 2: Load from File and Bytes](#3-exercise-2) | Multiple deserialization paths |
| 4 | [Exercise 3: Round-Trip Verification](#4-exercise-3) | Byte-level and semantic equivalence |
| 5 | [Exercise 4: External Data for Large Models](#5-exercise-4) | `save_as_external_data` workflow |
| 6 | [Exercise 5: Memory-Efficient Loading](#6-exercise-5) | Load graph structure without weights |
| 7 | [Exercise 6: Metadata Inspection Without Full Load](#7-exercise-6) | Lightweight model profiling |
| 8 | [Exercise 7: Model Size Analysis](#8-exercise-7) | Breakdown by component with visualization |
| 9 | [Exercise 8: Batch Model Processing](#9-exercise-8) | Directory-level operations |
| 10 | [Challenge: Model Registry with Save/Load](#10-challenge) | Versioned artifact management |
| 11 | [Summary](#11-summary) | Skills review |

In [ ]:
# 1. Setup <a id="1-setup"></a>
# !pip install onnx numpy matplotlib --quiet

import onnx
from onnx import helper, TensorProto, checker, numpy_helper, shape_inference
from onnx.external_data_helper import (
    convert_model_to_external_data,
    load_external_data_for_model,
)
import numpy as np
import os
import tempfile
import time
import copy
import hashlib
import json
import struct
import matplotlib.pyplot as plt

print(f"ONNX version: {onnx.__version__}")
print(f"IR version:   {onnx.IR_VERSION}")

In [ ]:
def build_mlp(hidden: int = 128, layers: int = 2, name: str = "mlp") -> onnx.ModelProto:
    """Build a multi-layer MLP with configurable size.
    
    Architecture: X -> [MatMul -> Add -> Relu] x (layers-1) -> MatMul -> Add -> Y
    Total parameters: layers * (hidden^2 + hidden)
    """
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", hidden])
    inits, nodes = [], []
    prev = "X"

    for i in range(layers):
        w = numpy_helper.from_array(
            np.random.randn(hidden, hidden).astype(np.float32) * 0.01, f"W{i}"
        )
        b = numpy_helper.from_array(np.zeros(hidden, dtype=np.float32), f"b{i}")
        inits.extend([w, b])
        mm_out = f"mm{i}"
        add_out = f"z{i}" if i < layers - 1 else "Y"
        nodes.append(helper.make_node("MatMul", [prev, f"W{i}"], [mm_out]))
        nodes.append(helper.make_node("Add", [mm_out, f"b{i}"], [add_out]))
        if i < layers - 1:
            relu_out = f"a{i}"
            nodes.append(helper.make_node("Relu", [add_out], [relu_out]))
            prev = relu_out
        else:
            prev = add_out

    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", hidden])
    graph = helper.make_graph(nodes, name, [X], [Y], initializer=inits)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.producer_name = "tutorial_io"
    model.model_version = 1
    checker.check_model(model)
    return model


# Quick verification
test_model = build_mlp(128, 2)
n_params = sum(int(np.prod(list(i.dims))) for i in test_model.graph.initializer)
print(f"Test model: {len(test_model.graph.node)} nodes, {n_params:,} parameters")
print(f"Serialized size: {len(test_model.SerializeToString()):,} bytes")

## 2. Exercise 1: Build and Save Models <a id="2-exercise-1"></a>

Practice the fundamental operation of constructing an ONNX model in memory and persisting it to disk.

**Key APIs:**
- `onnx.save(model, path)` — save to file
- `model.SerializeToString()` — serialize to bytes

The serialized format is Protocol Buffers binary. The file size relates to the model as:

$$\text{file\_size} \approx \text{graph\_structure} + \sum_{i=1}^{K} |\mathbf{W}_i| \cdot \text{sizeof}(\text{dtype}_i)$$

For float32 weights: $|\mathbf{W}_i| \cdot 4$ bytes per tensor.

In [ ]:
with tempfile.TemporaryDirectory(prefix="onnx_ex1_") as tmpdir:
    # Build models of increasing size
    configs = [(64, 1), (128, 2), (256, 3), (512, 2)]
    saved_files = []

    for hidden, layers in configs:
        model = build_mlp(hidden, layers, name=f"mlp_h{hidden}_l{layers}")
        path = os.path.join(tmpdir, f"model_h{hidden}_l{layers}.onnx")

        # Method 1: onnx.save
        onnx.save(model, path)
        fsize = os.path.getsize(path)
        saved_files.append((path, fsize, hidden, layers))

        # Method 2: raw bytes to file
        raw = model.SerializeToString()
        bytes_path = path.replace(".onnx", "_bytes.onnx")
        with open(bytes_path, "wb") as f:
            f.write(raw)

        # Verify both methods produce identical files
        assert os.path.getsize(bytes_path) == fsize, "File sizes must match"

    # Summary table
    print(f"{'Config':<18} | {'Params':>10} | {'File Size':>12} | {'Bytes/Param':>11}")
    print("-" * 60)
    for path, fsize, h, l in saved_files:
        n_params = l * (h * h + h)
        bpp = fsize / n_params
        print(f"h={h:>3}, l={l:<9} | {n_params:>10,} | {fsize:>10,} B | {bpp:>9.2f}")

    # Assertion: overhead ratio decreases with model size
    ratios = [fsize / (l * (h * h + h) * 4) for _, fsize, h, l in saved_files]
    assert ratios[-1] < ratios[0], "Larger models should have lower overhead ratio"
    print(f"\nOverhead ratios (file/raw_weights): {[f'{r:.3f}' for r in ratios]}")
    print("Larger models have proportionally less protobuf overhead. ✓")

## 3. Exercise 2: Load from File and Bytes <a id="3-exercise-2"></a>

ONNX provides multiple loading paths:

| Method | Input | Use Case |
|--------|-------|----------|
| `onnx.load(path)` | File path | Standard file loading |
| `onnx.load_model_from_string(bytes)` | Raw bytes | Network transfer, in-memory |
| `onnx.load(path, load_external_data=False)` | File path | Inspect structure only |
| `ModelProto.ParseFromString(bytes)` | Raw bytes | Low-level protobuf API |

In [ ]:
with tempfile.TemporaryDirectory(prefix="onnx_ex2_") as tmpdir:
    original = build_mlp(256, 2)
    path = os.path.join(tmpdir, "test_load.onnx")
    onnx.save(original, path)
    raw_bytes = original.SerializeToString()

    # Method 1: Load from file path
    t0 = time.perf_counter()
    m1 = onnx.load(path)
    t1 = time.perf_counter()
    print(f"1. onnx.load(path):              {(t1-t0)*1000:.2f} ms, {len(m1.graph.node)} nodes")

    # Method 2: Load from bytes
    t0 = time.perf_counter()
    m2 = onnx.load_model_from_string(raw_bytes)
    t1 = time.perf_counter()
    print(f"2. load_model_from_string:       {(t1-t0)*1000:.2f} ms, {len(m2.graph.node)} nodes")

    # Method 3: Low-level ParseFromString
    t0 = time.perf_counter()
    m3 = onnx.ModelProto()
    m3.ParseFromString(raw_bytes)
    t1 = time.perf_counter()
    print(f"3. ParseFromString (protobuf):   {(t1-t0)*1000:.2f} ms, {len(m3.graph.node)} nodes")

    # Method 4: Load from open file handle
    t0 = time.perf_counter()
    with open(path, "rb") as f:
        m4 = onnx.load(f)
    t1 = time.perf_counter()
    print(f"4. onnx.load(file_handle):       {(t1-t0)*1000:.2f} ms, {len(m4.graph.node)} nodes")

    # Verify all produce the same model
    ref = original.SerializeToString()
    assert m1.SerializeToString() == ref
    assert m2.SerializeToString() == ref
    assert m3.SerializeToString() == ref
    assert m4.SerializeToString() == ref
    print("\nAll 4 methods produce byte-identical models. ✓")

## 4. Exercise 3: Round-Trip Verification <a id="4-exercise-3"></a>

A **round-trip** test verifies that saving and reloading a model preserves:
1. **Byte identity**: `SerializeToString(loaded) == SerializeToString(original)`
2. **Structural equivalence**: same nodes, initializers, I/O
3. **Semantic equivalence**: $\forall \mathbf{x}: \|f_{\text{orig}}(\mathbf{x}) - f_{\text{loaded}}(\mathbf{x})\|_\infty = 0$

This is the gold standard test for any serialization pipeline.

In [ ]:
def roundtrip_test(model: onnx.ModelProto, label: str = "model") -> dict:
    """Comprehensive round-trip verification."""
    results = {"label": label}

    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "rt.onnx")
        original_bytes = model.SerializeToString()

        # Save
        onnx.save(model, path)
        results["file_size"] = os.path.getsize(path)

        # Load
        loaded = onnx.load(path)
        loaded_bytes = loaded.SerializeToString()

        # Check 1: Byte identity
        results["byte_identical"] = (original_bytes == loaded_bytes)

        # Check 2: Structural
        results["same_nodes"] = len(model.graph.node) == len(loaded.graph.node)
        results["same_inits"] = len(model.graph.initializer) == len(loaded.graph.initializer)

        # Check 3: Hash
        orig_hash = hashlib.sha256(original_bytes).hexdigest()[:16]
        load_hash = hashlib.sha256(loaded_bytes).hexdigest()[:16]
        results["hash_match"] = (orig_hash == load_hash)
        results["hash"] = orig_hash

        # Check 4: Semantic (via reference evaluator)
        try:
            from onnx.reference import ReferenceEvaluator
            dim = model.graph.input[0].type.tensor_type.shape.dim[1].dim_value
            x = np.random.randn(4, dim).astype(np.float32)
            y_orig = ReferenceEvaluator(model).run(None, {"X": x})[0]
            y_load = ReferenceEvaluator(loaded).run(None, {"X": x})[0]
            results["max_diff"] = float(np.abs(y_orig - y_load).max())
            results["semantic_match"] = results["max_diff"] == 0.0
        except Exception as e:
            results["semantic_match"] = "skipped"
            results["max_diff"] = str(e)[:50]

    return results


# Run round-trip on multiple model sizes
print(f"{'Model':<18} | {'Bytes':>7} | {'Byte ID':>7} | {'Hash':>7} | {'Semantic':>8} | SHA256")
print("-" * 80)

for h, l in [(64, 1), (128, 2), (256, 3), (512, 2)]:
    m = build_mlp(h, l)
    r = roundtrip_test(m, f"h{h}_l{l}")
    sem = "✓" if r["semantic_match"] is True else str(r["semantic_match"])
    print(
        f"h={h:>3}, l={l:<9} | {r['file_size']:>5,} | "
        f"{'✓' if r['byte_identical'] else '✗':>7} | "
        f"{'✓' if r['hash_match'] else '✗':>7} | "
        f"{sem:>8} | {r['hash']}"
    )

    assert r["byte_identical"], f"Round-trip failed for {r['label']}"

print("\nAll round-trip tests passed. ✓")

## 5. Exercise 4: External Data for Large Models <a id="5-exercise-4"></a>

When models exceed ~2 GB (protobuf size limit), weights must be stored externally.

**Workflow:**
```
model.onnx (graph only, ~KB)  +  weights.bin (tensor data, ~GB)
```

Key parameters for `onnx.save`:
- `save_as_external_data=True` — enable external storage
- `all_tensors_to_one_file=True` — single weights file
- `location="weights.bin"` — external file name
- `size_threshold=1024` — minimum tensor size to externalize (bytes)

In [ ]:
with tempfile.TemporaryDirectory(prefix="onnx_ext_") as tmpdir:
    model = build_mlp(512, 4)
    total_params = sum(int(np.prod(list(i.dims))) for i in model.graph.initializer)
    print(f"Model: {len(model.graph.node)} nodes, {total_params:,} params")

    # Save inline (standard)
    inline_path = os.path.join(tmpdir, "inline.onnx")
    onnx.save(model, inline_path)
    inline_size = os.path.getsize(inline_path)

    # Save with external data
    ext_path = os.path.join(tmpdir, "external.onnx")
    onnx.save(
        model,
        ext_path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.bin",
        size_threshold=64,
    )
    ext_onnx_size = os.path.getsize(ext_path)
    weights_path = os.path.join(tmpdir, "weights.bin")
    weights_size = os.path.getsize(weights_path)

    # Save with threshold (only large tensors go external)
    ext2_path = os.path.join(tmpdir, "partial_ext.onnx")
    onnx.save(
        model,
        ext2_path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights_partial.bin",
        size_threshold=4096,  # only tensors > 4KB go external
    )
    ext2_onnx = os.path.getsize(ext2_path)
    ext2_weights = os.path.getsize(os.path.join(tmpdir, "weights_partial.bin"))

    print(f"\n{'Format':<22} | {'Graph':>10} | {'Weights':>10} | {'Total':>10} | {'Graph %':>7}")
    print("-" * 70)
    print(f"{'Inline':<22} | {inline_size:>8,} B | {'(inline)':>10} | {inline_size:>8,} B | {'100%':>7}")
    print(f"{'External (all)':<22} | {ext_onnx_size:>8,} B | {weights_size:>8,} B | {ext_onnx_size+weights_size:>8,} B | {ext_onnx_size/(ext_onnx_size+weights_size)*100:>5.1f}%")
    print(f"{'External (>4KB)':<22} | {ext2_onnx:>8,} B | {ext2_weights:>8,} B | {ext2_onnx+ext2_weights:>8,} B | {ext2_onnx/(ext2_onnx+ext2_weights)*100:>5.1f}%")

    # Verify files on disk
    print(f"\nFiles in directory: {sorted(os.listdir(tmpdir))}")

    # Verify semantic equivalence after reload
    loaded_ext = onnx.load(ext_path)
    checker.check_model(loaded_ext)
    assert len(loaded_ext.graph.node) == len(model.graph.node)
    print(f"\nExternal-data model reloads and validates correctly. ✓")

    # Verify weights match
    for orig_init, ext_init in zip(model.graph.initializer, loaded_ext.graph.initializer):
        orig_arr = numpy_helper.to_array(orig_init)
        ext_arr = numpy_helper.to_array(ext_init)
        assert np.array_equal(orig_arr, ext_arr), f"Mismatch in {orig_init.name}"
    print("All weight tensors match exactly after external-data round-trip. ✓")

## 6. Exercise 5: Memory-Efficient Loading <a id="6-exercise-5"></a>

For large models, loading all weights into memory may be impractical just to inspect the graph.

**Strategy:** `onnx.load(path, load_external_data=False)` loads the graph skeleton but leaves
tensor data as references. This works for both inline and external-data models.

Use cases:
- Quick architecture inspection
- Building model catalogs
- Validating graph structure before committing to full load

In [ ]:
def inspect_architecture(model: onnx.ModelProto) -> dict:
    """Extract architecture info without needing weight data."""
    g = model.graph
    init_names = {i.name for i in g.initializer}
    real_inputs = [i for i in g.input if i.name not in init_names]

    info = {
        "graph_name": g.name,
        "n_nodes": len(g.node),
        "n_initializers": len(g.initializer),
        "op_types": sorted(set(n.op_type for n in g.node)),
        "inputs": [(i.name, [d.dim_param or d.dim_value for d in i.type.tensor_type.shape.dim]) for i in real_inputs],
        "outputs": [(o.name, [d.dim_param or d.dim_value for d in o.type.tensor_type.shape.dim]) for o in g.output],
        "initializer_shapes": {i.name: list(i.dims) for i in g.initializer},
    }
    return info


with tempfile.TemporaryDirectory() as tmpdir:
    large_model = build_mlp(1024, 4)
    path = os.path.join(tmpdir, "large.onnx")
    onnx.save(large_model, path)
    full_size = os.path.getsize(path)
    print(f"Model on disk: {full_size:,} bytes ({full_size/1024/1024:.2f} MB)")

    # Full load — measures time and memory
    t0 = time.perf_counter()
    full = onnx.load(path)
    t_full = (time.perf_counter() - t0) * 1000
    full_mem = len(full.SerializeToString())

    # Skeleton load
    t0 = time.perf_counter()
    skeleton = onnx.load(path, load_external_data=False)
    t_skel = (time.perf_counter() - t0) * 1000
    skel_mem = len(skeleton.SerializeToString())

    print(f"\n{'Method':<20} | {'Time (ms)':>10} | {'Memory (bytes)':>15} | {'Ratio':>6}")
    print("-" * 60)
    print(f"{'Full load':<20} | {t_full:>10.2f} | {full_mem:>13,} | {'1.00x':>6}")
    print(f"{'Skeleton load':<20} | {t_skel:>10.2f} | {skel_mem:>13,} | {skel_mem/full_mem:>5.3f}x")

    # Architecture inspection works on skeleton
    arch = inspect_architecture(skeleton)
    print(f"\nArchitecture from skeleton:")
    print(f"  Graph: {arch['graph_name']}")
    print(f"  Nodes: {arch['n_nodes']}")
    print(f"  Ops:   {arch['op_types']}")
    print(f"  Input: {arch['inputs']}")
    print(f"  Output: {arch['outputs']}")
    print(f"  Weight shapes: {list(arch['initializer_shapes'].items())[:4]}...")

    # Deferred loading: load weights on demand
    load_external_data_for_model(skeleton, tmpdir)
    restored_mem = len(skeleton.SerializeToString())
    assert restored_mem == full_mem, "After loading external data, size must match"
    print(f"\nAfter deferred load: {restored_mem:,} bytes (matches full load). ✓")

## 7. Exercise 6: Metadata Inspection Without Full Load <a id="7-exercise-6"></a>

Build a lightweight profiler that extracts key metadata from a model file without loading the entire
weight payload. This is essential for model catalogs and CI/CD pipelines.

**Goal:** Extract producer info, opset, graph name, node count, and parameter count
using only the skeleton load.

In [ ]:
def quick_profile(path: str) -> dict:
    """Profile an ONNX model file without loading weights."""
    model = onnx.load(path, load_external_data=False)
    g = model.graph

    # Parameter count from shape metadata (no weight data needed)
    total_params = sum(int(np.prod(list(i.dims))) for i in g.initializer)
    weight_bytes = total_params * 4  # assume float32

    return {
        "file": os.path.basename(path),
        "file_size": os.path.getsize(path),
        "producer": model.producer_name,
        "model_version": model.model_version,
        "ir_version": model.ir_version,
        "opset": model.opset_import[0].version,
        "graph_name": g.name,
        "n_nodes": len(g.node),
        "n_initializers": len(g.initializer),
        "total_params": total_params,
        "est_weight_MB": weight_bytes / 1024 / 1024,
        "op_types": sorted(set(n.op_type for n in g.node)),
        "metadata": {p.key: p.value for p in model.metadata_props},
    }


with tempfile.TemporaryDirectory() as tmpdir:
    # Create a suite of test models with metadata
    for h, l, prod in [(128, 2, "pytorch"), (256, 3, "tensorflow"), (512, 2, "jax")]:
        m = build_mlp(h, l)
        m.producer_name = prod
        m.model_version = l
        entry = m.metadata_props.add()
        entry.key, entry.value = "accuracy", f"{np.random.uniform(0.85, 0.99):.3f}"
        onnx.save(m, os.path.join(tmpdir, f"{prod}_h{h}_l{l}.onnx"))

    # Profile all models
    print(f"{'File':<28} | {'Producer':>10} | {'Params':>10} | {'Nodes':>5} | {'OpSet':>5} | {'Size':>10}")
    print("-" * 85)

    for fname in sorted(os.listdir(tmpdir)):
        if fname.endswith(".onnx"):
            p = quick_profile(os.path.join(tmpdir, fname))
            print(
                f"{p['file']:<28} | {p['producer']:>10} | {p['total_params']:>10,} | "
                f"{p['n_nodes']:>5} | {p['opset']:>5} | {p['file_size']:>8,} B"
            )

    print("\nAll profiled without loading weight data into memory. ✓")

## 8. Exercise 7: Model Size Analysis <a id="8-exercise-7"></a>

Break down model file size by component. For a model with $K$ initializers:

$$\text{total} = \underbrace{\text{graph\_proto}_{\text{overhead}}}_{\text{nodes, edges, names}} + \sum_{k=1}^{K} \underbrace{\left(\text{proto\_tag}_k + |\mathbf{W}_k| \cdot b_k\right)}_{\text{per-tensor cost}}$$

where $b_k$ is bytes per element (4 for float32, 2 for float16, 1 for int8).

In [ ]:
def detailed_size_analysis(model: onnx.ModelProto) -> dict:
    """Compute detailed size breakdown."""
    total = len(model.SerializeToString())

    # Per-initializer sizes
    tensor_sizes = {}
    for init in model.graph.initializer:
        arr = numpy_helper.to_array(init)
        tensor_sizes[init.name] = {
            "shape": list(init.dims),
            "elements": int(np.prod(list(init.dims))),
            "dtype": str(arr.dtype),
            "raw_bytes": arr.nbytes,
        }

    total_weight_bytes = sum(v["raw_bytes"] for v in tensor_sizes.values())
    graph_overhead = total - total_weight_bytes

    return {
        "total": total,
        "graph_overhead": graph_overhead,
        "weight_bytes": total_weight_bytes,
        "tensors": tensor_sizes,
        "overhead_pct": graph_overhead / total * 100,
        "weight_pct": total_weight_bytes / total * 100,
    }


# Analyze models of different sizes
models_to_analyze = [
    ("Small (64x1)", build_mlp(64, 1)),
    ("Medium (256x3)", build_mlp(256, 3)),
    ("Large (512x4)", build_mlp(512, 4)),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

names, totals, overheads, weights = [], [], [], []
for label, model in models_to_analyze:
    analysis = detailed_size_analysis(model)
    names.append(label)
    totals.append(analysis["total"])
    overheads.append(analysis["graph_overhead"])
    weights.append(analysis["weight_bytes"])
    print(f"{label}:")
    print(f"  Total: {analysis['total']:>10,} bytes")
    print(f"  Graph: {analysis['graph_overhead']:>10,} bytes ({analysis['overhead_pct']:.1f}%)")
    print(f"  Weight:{analysis['weight_bytes']:>10,} bytes ({analysis['weight_pct']:.1f}%)")
    print(f"  Tensors: {len(analysis['tensors'])}")
    print()

# Stacked bar chart
x = np.arange(len(names))
axes[0].bar(x, overheads, label="Graph overhead", color="#FF9800")
axes[0].bar(x, weights, bottom=overheads, label="Weight data", color="#2196F3")
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, fontsize=9)
axes[0].set_ylabel("Bytes")
axes[0].set_title("Size Breakdown", fontweight="bold")
axes[0].legend()

# Pie chart for largest model
axes[1].pie(
    [overheads[-1], weights[-1]],
    labels=["Graph", "Weights"],
    autopct="%1.1f%%",
    colors=["#FF9800", "#2196F3"],
    startangle=90,
)
axes[1].set_title(f"{names[-1]}\n({totals[-1]:,} B)", fontweight="bold")

# Per-tensor breakdown of largest model
analysis = detailed_size_analysis(models_to_analyze[-1][1])
sorted_tensors = sorted(analysis["tensors"].items(), key=lambda x: -x[1]["raw_bytes"])
t_names = [n for n, _ in sorted_tensors[:8]]
t_sizes = [v["raw_bytes"] / 1024 for _, v in sorted_tensors[:8]]
axes[2].barh(t_names, t_sizes, color="#4CAF50")
axes[2].set_xlabel("KB")
axes[2].set_title("Per-Tensor (Top 8)", fontweight="bold")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Exercise 8: Batch Model Processing <a id="9-exercise-8"></a>

Build a batch processor that operates on directories of ONNX files—validating, profiling,
and summarizing each model automatically.

In [ ]:
def batch_validate_and_profile(directory: str) -> list:
    """Validate and profile all .onnx files in a directory."""
    results = []
    onnx_files = sorted(f for f in os.listdir(directory) if f.endswith(".onnx"))

    for fname in onnx_files:
        path = os.path.join(directory, fname)
        entry = {"file": fname, "size": os.path.getsize(path)}

        try:
            model = onnx.load(path)
            entry["loaded"] = True
            entry["nodes"] = len(model.graph.node)
            entry["params"] = sum(int(np.prod(list(i.dims))) for i in model.graph.initializer)
            entry["ops"] = sorted(set(n.op_type for n in model.graph.node))

            try:
                checker.check_model(model)
                entry["valid"] = True
            except Exception as e:
                entry["valid"] = False
                entry["error"] = str(e)[:60]
        except Exception as e:
            entry["loaded"] = False
            entry["error"] = str(e)[:60]

        results.append(entry)

    return results


with tempfile.TemporaryDirectory() as tmpdir:
    # Generate test suite
    configs = [(64, 1), (128, 2), (256, 2), (256, 3), (512, 2), (128, 4)]
    for h, l in configs:
        m = build_mlp(h, l)
        onnx.save(m, os.path.join(tmpdir, f"mlp_h{h}_l{l}.onnx"))

    # Also create one invalid model for testing
    bad = build_mlp(64, 1)
    bad.graph.node[0].op_type = "FakeOp"
    # Save raw bytes (bypasses checker)
    with open(os.path.join(tmpdir, "invalid_model.onnx"), "wb") as f:
        f.write(bad.SerializeToString())

    # Run batch processing
    results = batch_validate_and_profile(tmpdir)

    print(f"{'File':<24} | {'Valid':>5} | {'Nodes':>5} | {'Params':>10} | {'Size':>10}")
    print("-" * 65)
    for r in results:
        if r.get("loaded"):
            v = "✓" if r.get("valid") else "✗"
            print(f"{r['file']:<24} | {v:>5} | {r['nodes']:>5} | {r['params']:>10,} | {r['size']:>8,} B")
        else:
            print(f"{r['file']:<24} | LOAD FAIL: {r.get('error', '')}")

    valid_count = sum(1 for r in results if r.get("valid"))
    print(f"\nSummary: {valid_count}/{len(results)} models valid")

    # Visualization
    valid_results = [r for r in results if r.get("loaded") and r.get("valid")]
    if valid_results:
        fig, ax = plt.subplots(figsize=(10, 4))
        fnames = [r["file"].replace(".onnx", "") for r in valid_results]
        sizes = [r["size"] / 1024 for r in valid_results]
        params = [r["params"] for r in valid_results]

        bars = ax.bar(range(len(fnames)), sizes, color="#2196F3", alpha=0.8)
        ax.set_xticks(range(len(fnames)))
        ax.set_xticklabels(fnames, rotation=35, ha="right", fontsize=8)
        ax.set_ylabel("File Size (KB)")
        ax.set_title("Batch Model Size Comparison", fontweight="bold")

        for bar, p in zip(bars, params):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f"{p:,}p", ha="center", va="bottom", fontsize=7)
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

## 10. Challenge: Model Registry with Save/Load <a id="10-challenge"></a>

Build a complete model registry system that:
1. Saves model versions with metadata (accuracy, training config)
2. Loads any version by name + version number
3. Maintains an index file for fast lookups
4. Supports comparing versions
5. Computes checksums for integrity verification

This simulates a production ML-ops pattern.

In [ ]:
class ModelRegistry:
    """Versioned model registry with metadata tracking and integrity checks."""

    def __init__(self, base_dir: str):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)
        self.index_path = os.path.join(base_dir, "index.json")
        self.index = self._load_index()

    def _load_index(self) -> dict:
        if os.path.exists(self.index_path):
            with open(self.index_path) as f:
                return json.load(f)
        return {}

    def _save_index(self):
        with open(self.index_path, "w") as f:
            json.dump(self.index, f, indent=2)

    def register(self, name: str, version: int, model: onnx.ModelProto,
                 metadata: dict = None):
        """Register a model version."""
        model.model_version = version
        if metadata:
            for k, v in metadata.items():
                entry = model.metadata_props.add()
                entry.key, entry.value = k, str(v)

        fname = f"{name}_v{version}.onnx"
        path = os.path.join(self.base_dir, fname)
        onnx.save(model, path)

        raw = model.SerializeToString()
        checksum = hashlib.sha256(raw).hexdigest()

        if name not in self.index:
            self.index[name] = []

        self.index[name].append({
            "version": version,
            "file": fname,
            "size": os.path.getsize(path),
            "checksum": checksum,
            "n_params": sum(int(np.prod(list(i.dims))) for i in model.graph.initializer),
            "n_nodes": len(model.graph.node),
            "opset": model.opset_import[0].version,
            "metadata": metadata or {},
        })
        self._save_index()
        return checksum

    def load(self, name: str, version: int = None) -> onnx.ModelProto:
        """Load a model by name and optional version (default=latest)."""
        if name not in self.index:
            raise KeyError(f"Model '{name}' not found")
        versions = self.index[name]
        if version is None:
            entry = max(versions, key=lambda x: x["version"])
        else:
            entry = next((v for v in versions if v["version"] == version), None)
            if entry is None:
                raise KeyError(f"Version {version} not found for '{name}'")

        path = os.path.join(self.base_dir, entry["file"])
        model = onnx.load(path)

        # Integrity check
        actual_checksum = hashlib.sha256(model.SerializeToString()).hexdigest()
        assert actual_checksum == entry["checksum"], "Checksum mismatch!"
        return model

    def list_versions(self, name: str) -> list:
        return sorted(self.index.get(name, []), key=lambda x: x["version"])

    def compare(self, name: str, v1: int, v2: int) -> dict:
        """Compare two versions of a model."""
        entries = {e["version"]: e for e in self.index[name]}
        e1, e2 = entries[v1], entries[v2]
        return {
            "size_delta": e2["size"] - e1["size"],
            "param_delta": e2["n_params"] - e1["n_params"],
            "node_delta": e2["n_nodes"] - e1["n_nodes"],
            "v1_meta": e1["metadata"],
            "v2_meta": e2["metadata"],
        }

    def summary(self):
        print(f"\n{'Model':<15} | {'Version':>7} | {'Params':>10} | {'Size':>10} | {'OpSet':>5} | Metadata")
        print("-" * 80)
        for name, versions in self.index.items():
            for v in sorted(versions, key=lambda x: x["version"]):
                meta_str = ", ".join(f"{k}={v}" for k, v in list(v["metadata"].items())[:3])
                print(
                    f"{name:<15} | {v['version']:>7} | {v['n_params']:>10,} | "
                    f"{v['size']:>8,} B | {v['opset']:>5} | {meta_str}"
                )


# Demo: simulate model development lifecycle
with tempfile.TemporaryDirectory(prefix="registry_") as reg_dir:
    registry = ModelRegistry(reg_dir)

    # Version 1: small model, low accuracy
    m1 = build_mlp(128, 2)
    registry.register("sentiment", 1, m1, {"accuracy": "0.82", "epochs": "5"})

    # Version 2: bigger model, better accuracy
    m2 = build_mlp(256, 3)
    registry.register("sentiment", 2, m2, {"accuracy": "0.91", "epochs": "20"})

    # Version 3: same arch, more training
    m3 = build_mlp(256, 3)
    registry.register("sentiment", 3, m3, {"accuracy": "0.94", "epochs": "50"})

    # Another model family
    m_ner = build_mlp(512, 2)
    registry.register("ner", 1, m_ner, {"f1": "0.88", "dataset": "conll2003"})

    registry.summary()

    # Load and verify
    loaded = registry.load("sentiment", version=2)
    assert loaded.model_version == 2
    print(f"\nLoaded sentiment v2: {len(loaded.graph.node)} nodes ✓")

    latest = registry.load("sentiment")  # gets v3
    assert latest.model_version == 3
    print(f"Latest sentiment: v{latest.model_version} ✓")

    # Compare versions
    diff = registry.compare("sentiment", 1, 3)
    print(f"\nv1 → v3 comparison:")
    print(f"  Params: +{diff['param_delta']:,}")
    print(f"  Size:   +{diff['size_delta']:,} bytes")
    print(f"  Nodes:  +{diff['node_delta']}")
    print(f"  Accuracy: {diff['v1_meta'].get('accuracy')} → {diff['v2_meta'].get('accuracy')}")

## 11. Summary <a id="11-summary"></a>

| Exercise | Skill | Key API |
|----------|-------|---------|
| 1. Build & Save | Model construction and persistence | `onnx.save()`, `SerializeToString()` |
| 2. Load Methods | Multiple deserialization paths | `onnx.load()`, `load_model_from_string()` |
| 3. Round-Trip | Byte-level + semantic verification | SHA-256, `ReferenceEvaluator` |
| 4. External Data | Large model storage pattern | `save_as_external_data`, `size_threshold` |
| 5. Efficient Load | Graph-only loading | `load_external_data=False` |
| 6. Metadata Inspect | Lightweight profiling | Skeleton load + shape dims |
| 7. Size Analysis | Component-level breakdown | Per-initializer byte accounting |
| 8. Batch Process | Directory-level operations | Automated validation pipeline |
| Challenge | Model Registry | Versioned artifacts with checksums |

### Key Formulas

- File size: $\text{total} \approx \text{overhead} + \sum_k |\mathbf{W}_k| \cdot b_k$
- Round-trip identity: $\text{Serialize}(\text{Load}(\text{Save}(M))) \equiv \text{Serialize}(M)$
- Overhead ratio: $\rho = \frac{\text{graph\_bytes}}{\text{total\_bytes}} \to 0$ as model grows